In [1]:
import truststore
truststore.inject_into_ssl()
import pandas as pd
import folium as fl
import geopandas as geo
import altair as alt
from math import ceil
from shapely import line_merge
from shapely.ops import substring
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib.colors import ListedColormap, BoundaryNorm
from IPython.display import display
%matplotlib inline

In [2]:
cabecalhos_mapa = {
    "User-Agent": (
        "acidentes-prf-sudeste-2025/1.0 "
        "(+https://github.com/miguel624/acidentes-prf-sudeste-2025)"
    )
}

# Distribuição e gravidade dos acidentes nas rodovias federais do Sudeste em 2025

##### Autores: Miguel Vitor Pereira; Ricardo Rodrigues Tose; Daniel Furtado Nunes Rocha da Silva
##### Professor: João Lucas Rulff da Costa

## Introdução

Esse trabalho tem como objetivo investigar a relação entre a gravidade dos acidentes com o horário, tipo de acidente, condições da via e do tempo. Buscaremos identificar padrões que orientem as discussões.

Não é necessário nenhum conhecimento prévio sobre o tema. 

## Dados e método

Utilizaremos a base pública da Polícia Rodoviária Federal (PRF), obtida por download do arquivo datatran2025.csv, na versão agrupada por ocorrência. A base nacional consultada contém 72.529 registros e 30 colunas; selecionaremos Minas Gerais, Espírito Santo, Rio de Janeiro e São Paulo no ano de 2025. Para esta análise, consideraremos grave a ocorrência com ao menos um morto ou ferido grave. Verificaremos duplicidades, valores ausentes e categorias antes de calcular contagens e proporções. Sem dados de fluxo de veículos, os resultados descreverão os acidentes registrados, sem estimar o risco por viagem ou estabelecer causalidade.

Observação: Ao pesquisar, foi descoberto que a PRF usa como georreferenciamento o SIRGAS 2000. 

# Preparação do dataset

In [3]:
df = pd.read_csv(r"datatran2025.csv",encoding="latin1",sep=";",decimal=",")
sudeste = df[df["uf"].isin(["SP", "MG", "ES", "RJ"])].copy()
sudeste.isna().sum()
#Existem alguns valores nulos, iremos verificar o que são.

id                         0
data_inversa               0
dia_semana                 0
horario                    0
uf                         0
br                         0
km                         0
municipio                  0
causa_acidente             0
tipo_acidente              0
classificacao_acidente     0
fase_dia                   0
sentido_via                0
condicao_metereologica     0
tipo_pista                 0
tracado_via                0
uso_solo                   0
pessoas                    0
mortos                     0
feridos_leves              0
feridos_graves             0
ilesos                     0
ignorados                  0
feridos                    0
veiculos                   0
latitude                   0
longitude                  0
regional                   0
delegacia                 10
uop                       16
dtype: int64

In [4]:
sudeste[sudeste.isna().any(axis=1)][["delegacia","uop"]]

,delegacia,uop
22094,NaN,NaN
28458,NaN,NaN
30176,NaN,NaN
31329,NaN,NaN
35906,DEL08-RJ,NaN
36475,NaN,NaN
44428,DEL01-RJ,NaN
45077,NaN,NaN
45200,NaN,NaN
45241,NaN,NaN


Os dados nulos se limitam a quais delegacias e unidade operacional ocorreu o acidente, como todos os outros dados estão completos e essas duas colunas não são de extrema importância, optaremos por deixá-las no dataset e se caso fizermos algum cálculo mais a frente em que esses dados sejam importantes iremos tomar outras medidas de análise.

In [5]:
print("IDs ausentes:",sudeste["id"].isna().sum())
# Não existe dados de acidentes repetidos.
print("Linhas com id repetido:",sudeste["id"].duplicated(keep=False).sum())
# Não existe ids repetidos
print("Duplicatas completas:",sudeste.duplicated().sum())
# Não existe linhas idênticas

IDs ausentes: 0
Linhas com id repetido: 0
Duplicatas completas: 0


In [6]:
datas_checagem = pd.to_datetime(
    sudeste["data_inversa"],
    format="%Y-%m-%d",
    errors="coerce"
)

horarios_checagem = pd.to_datetime(
    sudeste["horario"],
    format="%H:%M:%S",
    errors="coerce"
)

nomes_dias = {
    0: "segunda-feira",
    1: "terça-feira",
    2: "quarta-feira",
    3: "quinta-feira",
    4: "sexta-feira",
    5: "sábado",
    6: "domingo"
}

dias_esperados = (
    datas_checagem.dt.dayofweek.map(nomes_dias)
)

verificacao_temporal = pd.Series({
    "Datas inválidas ou ausentes": (
        datas_checagem.isna().sum()
    ),

    "Datas válidas fora de 2025": (
        datas_checagem.notna()
        & datas_checagem.dt.year.ne(2025)
    ).sum(),

    "Horários inválidos ou ausentes": (
        horarios_checagem.isna().sum()
    ),

    "Dia da semana diferente da data": (
        datas_checagem.notna()
        & sudeste["dia_semana"].ne(dias_esperados)
    ).sum()
}, name="quantidade")

display(verificacao_temporal)

Datas inválidas ou ausentes        0
Datas válidas fora de 2025         0
Horários inválidos ou ausentes     0
Dia da semana diferente da data    0
Name: quantidade, dtype: int64

In [7]:
colunas_texto = sudeste.select_dtypes(
    include=["object", "string"]
).columns

resultados_textos = []

for coluna in colunas_texto:
    texto = sudeste[coluna].astype("string")

    resultados_textos.append({
        "coluna": coluna,

        "vazios_ou_so_espacos": (
            texto.str.strip().eq("").sum()
        ),

        "espacos_nas_pontas": (
            texto.ne(texto.str.strip()).sum()
        )
    })

verificacao_textos = pd.DataFrame(
    resultados_textos
).set_index("coluna")

display(verificacao_textos)

,vazios_ou_so_espacos,espacos_nas_pontas
coluna,,
data_inversa,0,0
dia_semana,0,0
horario,0,0
uf,0,0
municipio,0,0
causa_acidente,0,0
tipo_acidente,0,0
classificacao_acidente,0,0
fase_dia,0,0


In [8]:
informacoes_desconhecidas = pd.Series({
    "BR igual a zero": (
        sudeste["br"].eq(0).sum()
    ),

    "Condição meteorológica ignorada": (
        sudeste["condicao_metereologica"]
        .eq("Ignorado").sum()
    ),

    "Sentido da via não informado": (
        sudeste["sentido_via"]
        .eq("Não Informado").sum()
    )
}, name="quantidade")

display(informacoes_desconhecidas)

BR igual a zero                     47
Condição meteorológica ignorada    271
Sentido da via não informado        47
Name: quantidade, dtype: int64

In [9]:
br_0=sudeste["br"]==0
sentido_ignorado=sudeste["sentido_via"]=="Não Informado"
sudeste[sentido_ignorado & br_0][["br","sentido_via"]].count()

br             47
sentido_via    47
dtype: int64

Existe 47 rodovias que não foram devidamente informadas, tendo seu valor como BR 0, e todas esses registros compartilham do fato de não terem o sentido da via informado (todo registro que não informa o sentido da via também não contém a rodovia devidamente informada.). Além disso, temos 271 registros em que a condição meteorológica foi ignorada. Decidimos manter todos esses dados, e ao calcularmos uma métrica em que fosse necessário usá-los, tomaremos medidas de análise que proporcione a interpretação mais fiel.

In [10]:
divergencia_feridos = sudeste["feridos"].ne(
    sudeste["feridos_leves"]
    + sudeste["feridos_graves"]
)

print(
    "Divergências no total de feridos:",
    divergencia_feridos.sum()
)

Divergências no total de feridos: 0


In [11]:
colunas_situacao = [
    "mortos",
    "feridos_leves",
    "feridos_graves",
    "ilesos",
    "ignorados"
]

soma_situacao = sudeste[
    colunas_situacao
].sum(
    axis=1,
    min_count=len(colunas_situacao)
)

divergencia_pessoas = sudeste["pessoas"].ne(
    soma_situacao
)

display(pd.Series({
    "Registros verificados": len(sudeste),

    "Divergências a investigar": (
        divergencia_pessoas.sum()
    ),

    "Percentual de divergências": round(
        divergencia_pessoas.mean() * 100,
        2
    )
}))

Registros verificados         23323.0
Divergências a investigar       886.0
Percentual de divergências        3.8
dtype: float64

In [12]:
display(
    sudeste.loc[
        divergencia_pessoas,
        ["id", "uf", "pessoas"] + colunas_situacao
    ].head()
)

,id,uf,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados
42,652869,ES,4,0,0,1,2,4
45,652885,MG,3,0,1,0,1,2
51,652930,SP,8,0,1,1,5,2
61,652976,ES,4,0,1,0,2,3
74,653044,MG,4,0,1,0,2,4


### Conclusão da limpeza
A preparação incluiu verificações de estrutura, valores ausentes, duplicidades, validade temporal, categorias, contagens numéricas e consistência entre campos. No recorte de 23.323 ocorrências, não foram encontrados IDs ausentes ou repetidos, duplicatas completas, datas ou horários inválidos, divergências entre data e dia da semana, nem contagens negativas ou fracionárias.

Os valores ausentes restringem-se às colunas delegacia e UOP, que não são utilizadas nas perguntas propostas; por isso, os registros foram preservados. Os códigos de rodovia não identificada e de condição meteorológica desconhecida foram documentados e tratados conforme a variável necessária a cada análise.

Foram identificadas 886 ocorrências (3,80%) com divergência entre o total de pessoas e a soma das categorias de situação física, incluindo ignorados. Esses valores foram preservados e registrados como uma limitação a investigar, pois não há fundamento suficiente para determinar uma correção.

# Como os acidentes registrados se distribuem entre os estados e as rodovias do Sudeste?

In [ ]:
total_regional = sudeste["id"].nunique()

resumo_uf = (
    sudeste.groupby("uf")["id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("acidentes")
    .reset_index()
)

resumo_uf["percentual_total_regional"] = (
    resumo_uf["acidentes"] / total_regional * 100
)

nomes_estados = {
    "MG": "Minas Gerais",
    "RJ": "Rio de Janeiro",
    "SP": "São Paulo",
    "ES": "Espírito Santo"
}

resumo_uf["estado"] = resumo_uf["uf"].map(nomes_estados)

resumo_uf[
    ["estado", "acidentes", "percentual_total_regional"]
].round(2)

In [ ]:
limite_x = float(resumo_uf["acidentes"].max()) * 1.15

base = alt.Chart(resumo_uf).encode(
    x=alt.X(
        "acidentes:Q",
        title="Número de acidentes registrados",
        scale=alt.Scale(domain=[0, limite_x])
    ),
    y=alt.Y(
        "estado:N",
        title=None,
        sort="-x"
    )
)

barras = base.mark_bar(
    color="#08519c"
).encode(
    tooltip=[
        alt.Tooltip(
            "estado:N",
            title="Estado"
        ),
        alt.Tooltip(
            "acidentes:Q",
            title="Acidentes",
            format="d"
        ),
        alt.Tooltip(
            "percentual_total_regional:Q",
            title="Participação regional (%)",
            format=".2f"
        )
    ]
)

rotulos = base.mark_text(
    align="left",
    baseline="middle",
    dx=5,
    color="#333333"
).encode(
    text=alt.Text(
        "acidentes:Q",
        format="d"
    )
)

grafico_uf = (
    barras + rotulos
).properties(
    width=620,
    height=220,
    title=alt.TitleParams(
        text="Acidentes registrados pela PRF por estado",
        subtitle=[
            "Sudeste — 2025",
            "Fonte: PRF. Universo: 23.323 registros."
        ]
    )
).configure_view(
    stroke=None
)

with alt.renderers.enable("png", scale_factor=2):
    display(grafico_uf)

Minas Gerais reúne a maior quantidade de acidentes registrados no recorte, com 9.570 ocorrências, correspondentes a 41,03% do total regional. Em seguida aparecem Rio de Janeiro, com 6.428 registros (27,56%); São Paulo, com 4.683 (20,08%); e Espírito Santo, com 2.642 (11,33%). Esses valores descrevem a distribuição dos registros da PRF entre os estados.

## Tratamento de dados.

observamos que criar uma camada do mapa contendo as rodovias federais da região ao invés de só plotar os acidentes seria uma abordagem melhor. Enquanto estávamos analisando percebemos que as rodovias federais ficam em poucas regiões do sudeste, e deixar essa representação sem identificar as rodovias iria dificultar imensamente a interpretação do leitor acerca da intensidade dos acidentes em cada região. Por isso, utilizamos o ArcGIS, um sistema de informações geográficas, para coletar os dados das rodovias federais e colocar como uma malha no nosso mapa (esses dados foram feitos no início de 2025, então bate de encontro com o dataset que estamos trabalhando). Ao lidar com os dados do ArcGIS, inicialmente excluímos as rodovias que foram planejadas e iremos analisar também as rovodias em obra de implantação, pois, ao pesquisar a terminologia, essas rodovias podem estar sob um leito natural. Então para saber se ela é ou não passível de análise no nosso dataset iremos comparar ao plotá-los. 

OBSERVAÇÃO: Percebemos que havia apenas uma rodovia em estado de obra de implantação, portanto optamos por deixá-la inclusa na construção da camada das rodovias.

A malha delimita a rede considerada no cálculo. O mapa de fundo do OpenStreetMap oferece referências geográficas, enquanto os limites estaduais obtidos do IBGE permitem distinguir os territórios dos quatro estados.

Organizamos a malha por UF e BR, reunimos as linhas conectadas quando possível e dividimos cada linha resultante em segmentos de até 10 km. O segmento final de uma linha pode ser menor. Sua origem depende da geometria da linha e não corresponde necessariamente aos marcos quilométricos oficiais da rodovia.

Para cada acidente, procuramos a linha mais próxima com a mesma UF e o mesmo código de BR informado no registro. Aceitamos a associação quando a distância até essa linha era de até 200 metros e não havia empate de menor distância detectado pelo procedimento. Em seguida, usamos a posição projetada ao longo da linha para determinar o segmento correspondente.

O limite de 200 metros foi uma escolha exploratória do grupo, sem validação externa de precisão. Para cada segmento, contamos os acidentes associados e dividimos essa quantidade por sua extensão em quilômetros. Assim, o indicador representado pelas cores é o número de acidentes associados por quilômetro, calculado em segmentos de até 10 km.

In [ ]:
dados_mapa = sudeste.copy()

acidentes_geo = geo.GeoDataFrame(
    dados_mapa,
    geometry=geo.points_from_xy(
        dados_mapa["longitude"],
        dados_mapa["latitude"]
    ),
    crs="EPSG:4674"
)

In [ ]:
acidentes_figura = acidentes_geo.to_crs("EPSG:31983")

fig, ax = plt.subplots(figsize=(10, 9), dpi=120)

acidentes_figura.plot(
    ax=ax,
    color="#08519c",
    markersize=2,
    alpha=0.35,
    zorder=2,
    label="Acidentes registrados"
)

cx.add_basemap(
    ax,
    source=cx.providers.OpenStreetMap.Mapnik,
    headers=cabecalhos_mapa,
    crs=acidentes_figura.crs,
    zoom=6,
    zorder=0
)

ax.set_title(
    "Localização dos acidentes registrados pela PRF\n"
    "Sudeste — 2025"
)

ax.set_axis_off()
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

Nesse gráfico conseguimos observar os acidentes a partir do mapa, ele nos mostra que é limitado tirar conclusões sem termos ciência da localização de todas as rodovias federais.

In [ ]:
import requests

consulta = (
    "https://geo.infrasa.gov.br/server/rest/services/"
    "Hosted/Rodovias_Federais/FeatureServer/0/query"
)

parametros = {
    "where": "sg_uf IN ('MG', 'ES', 'RJ', 'SP')",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": 4326,
    "resultRecordCount": 20000,
    "f": "geojson"
}

resposta = requests.get(
    consulta,
    params=parametros,
    timeout=60
)

resposta.raise_for_status()
conteudo = resposta.json()

if "error" in conteudo:
    raise RuntimeError(conteudo["error"])

malha = geo.GeoDataFrame.from_features(
    conteudo["features"],
    crs="EPSG:4326"
)

In [ ]:
malha_analise = malha[
    (malha["ds_jurisdi"] == "Federal")
    & ~malha["ds_legenda"].isin([
        "Planejada"
    ])
].copy()

In [ ]:
malha_analise = malha_analise.rename(columns={
    "fid": "id_trecho",
    "sg_uf": "uf",
    "vl_br": "br",
    "vl_km_inic": "km_inicial",
    "vl_km_fina": "km_final",
    "vl_extensa": "extensao_km"
})

In [ ]:
malha_figura = malha_analise.to_crs("EPSG:31983")

fig, ax = plt.subplots(figsize=(10, 9), dpi=120)

malha_figura.plot(
    ax=ax,
    color="#555555",
    linewidth=2,
    zorder=2,
    label="Malha federal selecionada"
)

cx.add_basemap(
    ax,
    source=cx.providers.OpenStreetMap.Mapnik,
    headers=cabecalhos_mapa,
    crs=malha_figura.crs,
    zoom=6,
    zorder=0
)

ax.set_title(
    "Malha de rodovias federais selecionada\n"
    "Sudeste — referência do início de 2025"
)

ax.set_axis_off()
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

Aqui conseguimos observar todas as rodovias federais do sudeste no ano de 2025, e com isso temos certeza de que os acidentes contidos nos nossos dados não estão distribuídos em todas as rodovias, o que deixa explícito a necessidade de comparar a concentração usando uma malha das rodovias para uma melhor interpretação.

In [ ]:
crs_figura = "EPSG:31983"

malha_figura = malha_analise.to_crs(crs_figura)
acidentes_figura = acidentes_geo.to_crs(crs_figura)

fig, ax = plt.subplots(
    figsize=(10, 9),
    dpi=120
)

malha_figura.plot(
    ax=ax,
    color="#2F2828",
    linewidth=0.6,
    label="Rodovias federais"
)

acidentes_figura.plot(
    ax=ax,
    color="#08519c",
    markersize=2,
    alpha=0.35,
    label="Acidentes registrados"
)

cx.add_basemap(
    ax,
    source=cx.providers.OpenStreetMap.Mapnik,
    headers=cabecalhos_mapa,
    crs=malha_figura.crs,
    zoom=6,
    zorder=0
)

ax.set_title(
    "Acidentes nas rodovias federais do Sudeste — 2025"
)

ax.set_axis_off()
ax.legend(loc="lower left")

plt.tight_layout()
plt.show()

No mapa acima percebemos que existe alguns poucos casos em que os acidentes não estão acima de uma rodovia federal, mas a maioria esmagadora está de acordo com o mapa das rodovias que fizemos. A partir de agora, para fazermos nosso geográfico evidenciando a distribuição de acidentes na região sudeste, iremos fazer com que cada acidente esteja designado a uma parte da rodovia, para que dessa forma possamos calcular a concentração de acidentes em uma determinada distância por rodovia (iremos usar 10km como base).

In [ ]:
malha_base = malha_analise[
    ["uf", "br", "geometry"]
].copy()

acidentes_base = acidentes_geo[
    ["id", "uf", "br", "km", "municipio", "geometry"]
].copy()

for tabela in [malha_base, acidentes_base]:
    tabela["br"] = (
        tabela["br"]
        .astype(int)
        .astype(str)
        .str.zfill(3)
    )

#Pegamos apenas as colunas necessária e padronizamos os números da BR das tabelas.

A coluna "br" contém rodovias com 2 dígitos ou 1, sendo que por padrão as rodovias federais tem 3. Por isso, padronizamos as rodovias para terem 3 dígitos, preenchendo com zeros as rodovias que não contêm 3 dígitos.

In [ ]:
redes = {}
partes_mapa = []

for (uf, br), grupo in malha_base.groupby(["uf", "br"]):

    crs_metrico = grupo.estimate_utm_crs(
        datum_name="SIRGAS 2000"
    )

    rede = (
        grupo.to_crs(crs_metrico)
        .dissolve(by=["uf", "br"])
        .reset_index()
    )

    rede["geometry"] = rede.geometry.apply(line_merge)

    linhas = (
        rede.explode(index_parts=False)
        .reset_index(drop=True)
    )

    linhas["id_linha"] = [
        f"{uf}_{br}_{numero}"
        for numero in linhas.index
    ]

    redes[(uf, br)] = (crs_metrico, linhas)

    segmentos = []

    for linha in linhas.itertuples():

        comprimento = linha.geometry.length
        quantidade = ceil(comprimento / 10000)

        for numero in range(quantidade):

            inicio = numero * 10000
            fim = min(inicio + 10000, comprimento)

            pedaco = substring(
                linha.geometry,
                inicio,
                fim
            )

            segmentos.append({
                "id_segmento": f"{linha.id_linha}_{numero}",
                "uf": uf,
                "br": br,
                "extensao_km": pedaco.length / 1000,
                "geometry": pedaco
            })

    partes_geo = geo.GeoDataFrame(
        segmentos,
        geometry="geometry",
        crs=crs_metrico
    )

    partes_geo["geometry"] = partes_geo.geometry.simplify(10)

    partes_mapa.append(
        partes_geo.to_crs("EPSG:4326")
    )

segmentos_10km = geo.GeoDataFrame(
    pd.concat(partes_mapa, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

segmentos_10km.head()

In [ ]:
distancia_maxima = 200
resultados = []

for (uf, br), (crs_metrico, linhas) in redes.items():

    pontos = acidentes_base[
        (acidentes_base["uf"] == uf)
        & (acidentes_base["br"] == br)
    ].to_crs(crs_metrico)

    if pontos.empty:
        continue

    candidatos = geo.sjoin_nearest(
        pontos,
        linhas[["id_linha", "geometry"]],
        how="left",
        max_distance=distancia_maxima,
        distance_col="distancia_m"
    )

    multiplicidade = (
        candidatos.groupby("id")["id_linha"]
        .transform("size")
    )

    ids_ambiguos = candidatos.loc[
        multiplicidade > 1, "id"
    ].unique()

    for id_acidente in ids_ambiguos:
        resultados.append({
            "id": id_acidente,
            "id_segmento": None,
            "distancia_m": None,
            "status": "ambiguo"
        })

    linhas_por_id = linhas.set_index("id_linha").geometry

    for ponto in candidatos[
        multiplicidade == 1
    ].itertuples():

        id_segmento = None
        status = "distante"

        if pd.notna(ponto.id_linha):

            linha = linhas_por_id.loc[ponto.id_linha]

            posicao_m = linha.project(ponto.geometry)

            numero = min(
                int(posicao_m // 10000),
                ceil(linha.length / 10000) - 1
            )

            id_segmento = f"{ponto.id_linha}_{numero}"
            status = "associado"

        resultados.append({
            "id": ponto.id,
            "id_segmento": id_segmento,
            "distancia_m": ponto.distancia_m,
            "status": status
        })

associacoes = pd.DataFrame(resultados)

auditoria = acidentes_base.merge(
    associacoes,
    on="id",
    how="left",
    validate="one_to_one"
)

auditoria["status"] = auditoria["status"].fillna(
    "sem_malha_uf_br"
)

auditoria["status"].value_counts()

Usando nossos métodos, conseguimos associar 22008 acidentes do total que temos. 

In [ ]:
associados = auditoria[
    auditoria["status"] == "associado"
].copy()

contagens = (
    associados.groupby("id_segmento")["id"]
    .nunique()
)

segmentos_10km["acidentes"] = (
    segmentos_10km["id_segmento"]
    .map(contagens)
    .fillna(0)
    .astype(int)
)

segmentos_10km["acidentes_por_km"] = (
    segmentos_10km["acidentes"]
    / segmentos_10km["extensao_km"]
)

assert segmentos_10km["acidentes"].sum() == len(associados)

segmentos_10km[
    ["uf", "br", "extensao_km", "acidentes", "acidentes_por_km"]
].head()

In [ ]:
segmentos_10km.nlargest(
    10, "acidentes_por_km"
)[[
    "uf",
    "br",
    "extensao_km",
    "acidentes",
    "acidentes_por_km"
]]

In [ ]:
url_estados = (
    "https://servicodados.ibge.gov.br/"
    "api/v3/malhas/regioes/SE"
)

resposta_estados = requests.get(
    url_estados,
    params={
        "formato": "application/vnd.geo+json",
        "intrarregiao": "UF",
        "qualidade": "minima"
    },
    timeout=60
)

resposta_estados.raise_for_status()

estados = geo.GeoDataFrame.from_features(
    resposta_estados.json()["features"],
    crs="EPSG:4326"
)

estados["uf"] = estados["codarea"].astype(str).map({
    "31": "MG",
    "32": "ES",
    "33": "RJ",
    "35": "SP"
})

In [ ]:
segmentos_figura = segmentos_10km.to_crs(crs_figura)
estados_figura = estados.to_crs(crs_figura)

fig, ax = plt.subplots(
    figsize=(10, 9),
    dpi=120
)

estados_figura.boundary.plot(
    ax=ax,
    color="#555555",
    linewidth=0.8,
    zorder=2
)

segmentos_figura.plot(
    ax=ax,
    color="#999999",
    linewidth=1
)

com_acidentes = segmentos_figura[
    segmentos_figura["acidentes"] > 0
]

maximo = max(
    20.1,
    float(com_acidentes["acidentes_por_km"].max())
)

limites = [0, 1, 3, 5, 10, 20, maximo]

cores = ListedColormap([
    "#fed976",
    "#feb24c",
    "#fd8d3c",
    "#f03b20",
    "#bd0026",
    "#800026"
])

normalizacao = BoundaryNorm(
    limites,
    ncolors=cores.N,
    clip=True
)
com_acidentes.plot(
    ax=ax,
    column="acidentes_por_km",
    cmap=cores,
    norm=normalizacao,
    linewidth=2.5,
    zorder=3,
    legend=True,
    legend_kwds={
        "label": "Acidentes por km, calculados em segmentos de até 10 km",
        "ticks": limites,
        "spacing": "uniform",
        "shrink": 0.7
    }
)

cx.add_basemap(
    ax,
    source=cx.providers.OpenStreetMap.Mapnik,
    headers=cabecalhos_mapa,
    crs=segmentos_figura.crs,
    zoom=6,
    zorder=0
)

ax.set_title(
    "Concentração de acidentes nas rodovias federais\n"
    "Sudeste — 2025"
)

ax.set_axis_off()

fig.text(
    0.1, 0.02,
    "Cinza: segmentos sem registros associados."
)
for estado in estados_figura.itertuples():

    ponto_rotulo = estado.geometry.representative_point()

    ax.text(
        ponto_rotulo.x,
        ponto_rotulo.y,
        estado.uf,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
        zorder=4
    )

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## Conclusão:

Os acidentes registrados pela PRF no Sudeste em 2025 apresentam diferenças tanto nos totais por estado quanto na concentração local ao longo da malha. Minas Gerais reúne a maior parcela dos registros regionais com aproximadamente 41% dos registros de acidentes.

Ao analisar a concentração de acidentes nas rodovias federais do Sudeste, podemos perceber que existe uma quantidade considerável de segmentos sem registros associados. Mas de uma forma geral, os acidentes ficaram bem distribuídos nas rodovias. Entretanto, nas rodovias perto das capitais os acidentes aumentam grandemente. Além disso, na região litorânea de São Paulo e Rio de Janeiro as médias de acidentes são maiores que no restante dos estados.

# Quais meses, dias da semana e faixas horárias concentram mais ocorrências?

## Tratamento de Dados:
Enquanto estávamos coletando os dados referentes aos períodos que iremos analisar, percebemos uma inconsistência nos dados da coluna "fase_dia". Nessa coluna, mesmo as células que informavam que o acidente havia acontecido em pleno dia, existem registros variando de 00 até 23, o que não é condizente. Portanto, ao invés de usarmos essa coluna, nós mesmos definimos 4 períodos do dia e analisamos o horário em que cada acidente ocorreu. A maneira de analisar os períodos foi uma escolha exploratória do grupo, sem validação externa de precisão. Fora isso, nossos dados sobre os registros durante os meses foi criado usando o mesmo método de observação do horário e os dias da semana usamos os dados já fornecidos pelo dataset.

In [ ]:
meses={}
meses_ano=["Janeiro","Fevereiro","Março","Abril","Maio","Junho","Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]
dias_semana={}
horarios={}
for linha in range(sudeste["id"].nunique()):
    mes=sudeste.iloc[linha]["data_inversa"]
    mes=int(mes.split("-")[1])
    meses[meses_ano[mes-1]]=meses.get(meses_ano[mes-1],0) + 1

    dia=sudeste.iloc[linha]["dia_semana"]
    dias_semana[dia]=dias_semana.get(dia,0) + 1

    hora=sudeste.iloc[linha]["fase_dia"]
    horarios[hora]=horarios.get(hora,0) + 1

meses_final={"Mês":[],"Registros":[]}
for mes,quantidade in meses.items():
    meses_final["Mês"].append(mes)
    meses_final["Registros"].append(quantidade)

dias_semana_final={"Dia da semana":[],"Registros":[]}
for dia,quantidade in dias_semana.items():
    dias_semana_final["Dia da semana"].append(dia)
    dias_semana_final["Registros"].append(quantidade)

horarios_final={"Faixa horária":[],"Registros":[]}
for hora,quantidade in horarios.items():
    horarios_final["Faixa horária"].append(hora)
    horarios_final["Registros"].append(quantidade)
registros_mensais=pd.DataFrame(meses_final).sort_values(by="Registros",ascending=False)
registros_da_semana=pd.DataFrame(dias_semana_final).sort_values(by="Registros",ascending=False)
registros_faixa_horaria=pd.DataFrame(horarios_final).sort_values(by="Registros",ascending=False)


In [ ]:
registros_mensais

In [ ]:
registros_da_semana

In [ ]:
registros_faixa_horaria

In [ ]:
pleno_dia=sudeste["fase_dia"]=="Pleno dia"
horario_minimo_pleno_dia=set()
for linha in range(sudeste[pleno_dia]["id"].nunique()):
    horario=sudeste[pleno_dia].iloc[linha]["horario"]
    horario_minimo_pleno_dia.add(int(horario.split(":")[0]))


In [ ]:
print(horario_minimo_pleno_dia)

Aqui está a obvervação acerca da inconsistência dos dados do dataset.

In [ ]:
horarios={}
for linha in range(sudeste["id"].nunique()):
    hora=int(sudeste.iloc[linha]["horario"].split(":")[0])
    if hora>=18:
        faixa="Noite"
    elif hora>=12:
        faixa="Tarde"
    elif hora>=6:
        faixa="Manhã"
    else:
        faixa="Madrugada"
    horarios[faixa]=horarios.get(faixa,0) + 1

horarios_final={"Faixa horária":[],"Registros":[]}
for faixa,quantidade in horarios.items():
    horarios_final["Faixa horária"].append(faixa)
    horarios_final["Registros"].append(quantidade)
registros_faixa_horaria=pd.DataFrame(horarios_final).sort_values(by="Registros",ascending=False)

In [ ]:
registros_faixa_horaria

In [ ]:
registros_faixa_horaria["Percentual"]=round((registros_faixa_horaria["Registros"]/registros_faixa_horaria["Registros"].sum())*100,2)
registros_da_semana["Percentual"]=round((registros_da_semana["Registros"]/registros_da_semana["Registros"].sum())*100,2)
registros_mensais["Percentual"]=round((registros_mensais["Registros"]/registros_mensais["Registros"].sum())*100,2)

In [ ]:
ordem_meses = [
    "Janeiro", "Fevereiro", "Março", "Abril",
    "Maio", "Junho", "Julho", "Agosto",
    "Setembro", "Outubro", "Novembro", "Dezembro"
]

ordem_semana = [
    "segunda-feira", "terça-feira", "quarta-feira",
    "quinta-feira", "sexta-feira", "sábado", "domingo"
]

ordem_faixas = [
    "Madrugada", "Manhã", "Tarde", "Noite"
]


def criar_barras(tabela, coluna_categoria, ordem, titulo):

    limite_x = float(tabela["Registros"].max()) * 1.20

    base = alt.Chart(tabela).encode(
        x=alt.X(
            "Registros:Q",
            title="Acidentes registrados",
            scale=alt.Scale(domain=[0, limite_x])
        ),
        y=alt.Y(
            f"{coluna_categoria}:N",
            title=None,
            sort=ordem
        )
    )

    barras = base.mark_bar(
        color="#08519c",
        size=18
    ).encode(
        tooltip=[
            alt.Tooltip(
                f"{coluna_categoria}:N",
                title=coluna_categoria
            ),
            alt.Tooltip(
                "Registros:Q",
                title="Acidentes",
                format="d"
            ),
            alt.Tooltip(
                "Percentual:Q",
                title="Participação regional (%)",
                format=".2f"
            )
        ]
    )

    rotulos = base.mark_text(
        align="left",
        baseline="middle",
        dx=4,
        color="#333333"
    ).encode(
        text=alt.Text(
            "Registros:Q",
            format="d"
        )
    )

    return (barras + rotulos).properties(
        width=250,
        height=330,
        title=titulo
    )


grafico_meses = criar_barras(
    registros_mensais,
    "Mês",
    ordem_meses,
    "Meses"
)

grafico_semana = criar_barras(
    registros_da_semana,
    "Dia da semana",
    ordem_semana,
    "Dias da semana"
)

grafico_faixas = criar_barras(
    registros_faixa_horaria,
    "Faixa horária",
    ordem_faixas,
    "Faixas horárias"
)


painel_temporal = alt.hconcat(
    grafico_meses,
    grafico_semana,
    grafico_faixas,
    spacing=25
).resolve_scale(
    x="independent",
    y="independent"
).properties(
    title=alt.TitleParams(
        text="Distribuição temporal dos acidentes registrados pela PRF",
        subtitle=[
            "Sudeste — 2025 | 23.323 registros",
            "Cada painel possui sua própria escala horizontal.",
            "Faixas: 00h–06h, 06h–12h, 12h–18h e 18h–24h."
        ]
    )
).configure_view(
    stroke=None
).configure_axis(
    labelFontSize=11,
    titleFontSize=11
)


with alt.renderers.enable("png", scale_factor=2):
    display(painel_temporal)

## Conclusão
Podemos perceber que a partir de Abril os acidentes apenas aumentaram até dezembro, e além disso existe uma diferença de 504 acidentes entre o mês de maior (Dezembro, com 9,45%) e o de menor (Fevereiro, com 7,28%) registro. Quanto aos dias da semana, sexta-feira e os finais de semana contêm as maiores concentrações individuais de acidentes registrados, e a grande maioria ocorre no período da tarde, sendo 1,5 vezes a mais da quantidade de acidentes que ocorrem durante a madrugada.

# Quais tipos de acidente apresentam maior proporção de ocorrências graves?

## Tratamento de dados:
Assim como foi falado antes, iremos considerar uma ocorrência grave se houver pelo menos uma morte ou um ferido grave. Para fazermos essa análise, separamos os acidentes graves dos acidentes totais em um dataframe, após isso fizemos os cálculos de registros por tipo de acidente e também a quantidade de mortes totais por tipo de acidente. Para calcularmos a proporção de acidentes graves, comparamos os registros totais com os registros em que houveram mortes ou feridos graves, e adicionamos como complemento o total de mortos em cada tipo de acidente.

In [ ]:
graves=(sudeste["mortos"]>0) | (sudeste["feridos_graves"]>0)
acidentes_graves=sudeste[graves].copy()

In [ ]:
tipos_acidentes_graves={}
mortes_por_tipo_acidente={}
for linha in range(acidentes_graves["id"].count()):
    tipo=acidentes_graves.iloc[linha]["tipo_acidente"]
    tipos_acidentes_graves[tipo] = tipos_acidentes_graves.get(tipo,0) + 1
    mortes=acidentes_graves.iloc[linha]["mortos"]
    if mortes>0:
        mortes_por_tipo_acidente[tipo]=mortes_por_tipo_acidente.get(tipo,0) + mortes

In [ ]:
resumo_gravidade = (sudeste.groupby("tipo_acidente")["id"].nunique().rename("total_acidentes").reset_index())

resumo_gravidade["acidentes_graves"] = (resumo_gravidade["tipo_acidente"].map(tipos_acidentes_graves).fillna(0).astype(int))

resumo_gravidade["mortos"] = (resumo_gravidade["tipo_acidente"].map(mortes_por_tipo_acidente).fillna(0).astype(int))

resumo_gravidade["proporcao_graves"] = (resumo_gravidade["acidentes_graves"]/ resumo_gravidade["total_acidentes"])

resumo_gravidade = resumo_gravidade.sort_values("proporcao_graves",ascending=False)

resumo_gravidade["rotulo"] = [(f"{linha.proporcao_graves:.1%}"f"({linha.acidentes_graves}/{linha.total_acidentes})").replace(".", ",")for linha in resumo_gravidade.itertuples()]

resumo_gravidade

In [ ]:
base = alt.Chart(resumo_gravidade).encode(
    x=alt.X(
        "proporcao_graves:Q",
        title="Proporção de acidentes graves dentro de cada tipo",
        scale=alt.Scale(domain=[0, 1]),
        axis=alt.Axis(format=".0%")
    ),
    y=alt.Y(
        "tipo_acidente:N",
        title=None,
        sort="-x",
        axis=alt.Axis(labelLimit=280)
    )
)

barras = base.mark_bar(
    color="#08519c",
    size=18
).encode(
    tooltip=[
        alt.Tooltip(
            "tipo_acidente:N",
            title="Tipo de acidente"
        ),
        alt.Tooltip(
            "acidentes_graves:Q",
            title="Acidentes graves",
            format="d"
        ),
        alt.Tooltip(
            "total_acidentes:Q",
            title="Todos os acidentes do tipo",
            format="d"
        ),
        alt.Tooltip(
            "proporcao_graves:Q",
            title="Proporção de graves",
            format=".1%"
        ),
        alt.Tooltip(
            "mortos:Q",
            title="Pessoas mortas",
            format="d"
        )
    ]
)

rotulos = base.mark_text(
    align="left",
    baseline="middle",
    dx=5,
    color="#333333"
).encode(
    text="rotulo:N"
)

grafico_gravidade = (
    barras + rotulos
).properties(
    width=620,
    height=alt.Step(28),
    title=alt.TitleParams(
        text="Proporção de acidentes graves por tipo de acidente",
        subtitle=[
            "Sudeste — 2025 | Fonte: PRF",
            "Grave: pelo menos um morto ou um ferido grave.",
            "Rótulos: proporção (acidentes graves / total do tipo)."
        ]
    )
).configure_view(
    stroke=None
)

from IPython.display import display

with alt.renderers.enable("png", scale_factor=2):
    display(grafico_gravidade)

## Conclusão
Embora "Sinistro pessoal do trânsito" (qualquer evento que resulte em dano ao veículo, à sua carga, ou em lesões a pessoas e/ou animais, podendo causar prejuízos ao trânsito, à via ou ao meio ambiente. Esse termo enfatiza que essas ocorrências são previsíveis e evitáveis) tenha a maior proporção, existe apenas 4 registros desse tipo de acidente. Uma observação interessante é que, mesmo tendo tido 661 acidentes, não houve nenhum acidente grave em que houve incêndio. Vale ressaltar que, embora colisão frontal esteja em terceiro lugar, ele foi o tipo de acidente que mais causou mortes, portanto essa visualização não representa o potencial perigo de qualquer tipo de acidente, apenas a proporção dos acidentes graves registrados. Além disso, a proporção de acidentes graves por acidentes totais é de aproximadamente 25%.

# A proporção de acidentes graves difere entre dia e noite, em pistas simples e duplas?

## Tratamento de dados:
Consideramos dia como sendo o período entre 6:00 às 18:00, e fizemos nosso dataset contendo essa classificação da linha. Após isso, fizemos de forma individual os cálculos referentes apenas dos horários e apenas dos tipos de pistas, com o intuito de analisar quantas pistas simples e duplas existiam comparado com o total. Após isso, juntamos as classificações do tipo da pista com o horário do acidente, comparando cada caso com a quantidade de acidentes graves com os totais.

In [ ]:
sudeste_com_horarios = sudeste.copy()
horarios=[]
horarios_graves=[]
for linha in range(sudeste_com_horarios["id"].count()):
    hora=int(sudeste_com_horarios.iloc[linha]["horario"].split(":")[0])
    if hora>=6 and hora<18:
        horarios.append("Dia")
    else:
        horarios.append("Noite")
sudeste_com_horarios["faixa_horaria"]=horarios

for linha in range(acidentes_graves["id"].count()):
    hora=int(acidentes_graves.iloc[linha]["horario"].split(":")[0])
    if hora>=6 and hora<18:
        horarios_graves.append("Dia")
    else:
        horarios_graves.append("Noite")
acidentes_graves["faixa_horaria"]=horarios_graves

In [ ]:
acidentes_por_horario=sudeste_com_horarios.groupby("faixa_horaria")["id"].nunique().rename("total_acidentes").reset_index()
acidentes_por_horario["total_acidentes"]=acidentes_por_horario["total_acidentes"].astype(int)
graves_por_horario=acidentes_graves.groupby("faixa_horaria")["id"].nunique()
acidentes_por_horario["acidentes_graves"]=(acidentes_por_horario["faixa_horaria"].map(graves_por_horario).fillna(0).astype(int))

acidentes_por_horario["proporcao_graves"]=acidentes_por_horario["acidentes_graves"]/acidentes_por_horario["total_acidentes"]
acidentes_por_horario["rotulo"] = [(f"{linha.proporcao_graves:.1%}"f"({linha.acidentes_graves}/{linha.total_acidentes})").replace(".", ",")for linha in acidentes_por_horario.itertuples()]

acidentes_por_horario

In [ ]:
acidentes_por_pista=sudeste_com_horarios.groupby("tipo_pista")["id"].nunique().rename("total_acidentes").reset_index()
graves_por_pista=acidentes_graves.groupby("tipo_pista")["id"].nunique()
acidentes_por_pista["acidentes_graves"] = acidentes_por_pista["tipo_pista"].map(graves_por_pista).fillna(0).astype(int)
acidentes_por_pista["proporcao_graves"] = acidentes_por_pista["acidentes_graves"]/acidentes_por_pista["total_acidentes"]
acidentes_por_pista["rotulo"]= [(f"{linha.proporcao_graves:.1%}"f"({linha.acidentes_graves}/{linha.total_acidentes})").replace(".",",") for linha in acidentes_por_pista.itertuples()]
acidentes_por_pista

In [ ]:
grupos = ["tipo_pista", "faixa_horaria"]

totais_por_grupo = (sudeste_com_horarios.groupby(grupos)["id"].nunique().rename("total_acidentes").reset_index())

graves_por_grupo = (acidentes_graves.groupby(grupos)["id"].nunique().rename("acidentes_graves").reset_index())

comparacao_pista_horario = totais_por_grupo.merge(graves_por_grupo,on=grupos,how="left",validate="one_to_one")

comparacao_pista_horario["proporcao_graves"] = (comparacao_pista_horario["acidentes_graves"]/ comparacao_pista_horario["total_acidentes"])

comparacao_pista_horario = comparacao_pista_horario[comparacao_pista_horario["tipo_pista"].isin(["Simples", "Dupla"])].copy()

comparacao_pista_horario["rotulo"] = [(f"{linha.proporcao_graves:.1%}\n"f"({linha.acidentes_graves}/{linha.total_acidentes})").replace(".", ",")for linha in comparacao_pista_horario.itertuples()]

comparacao_pista_horario

In [ ]:
base_calor = alt.Chart(comparacao_pista_horario).encode(
    x=alt.X(
        "faixa_horaria:N",
        title="Período",
        sort=["Dia", "Noite"],
        axis=alt.Axis(
            orient="top",
            labelAngle=0,
            labelPadding=8
        )
    ),
    y=alt.Y(
        "tipo_pista:N",
        title="Tipo de pista",
        sort=["Simples", "Dupla"],
        axis=alt.Axis(
            labelAngle=0,
            labelPadding=8
        )
    )
)

celulas = base_calor.mark_rect(
    stroke="white",
    strokeWidth=3
).encode(
    color=alt.Color(
        "proporcao_graves:Q",
        title="Proporção de graves",
        scale=alt.Scale(
            scheme="blues",
            domain=[0, 0.4]
        ),
        legend=alt.Legend(
            format=".0%",
            values=[0, 0.1, 0.2, 0.3, 0.4]
        )
    ),
    tooltip=[
        alt.Tooltip("tipo_pista:N", title="Tipo de pista"),
        alt.Tooltip("faixa_horaria:N", title="Período"),
        alt.Tooltip(
            "acidentes_graves:Q",
            title="Acidentes graves",
            format="d"
        ),
        alt.Tooltip(
            "total_acidentes:Q",
            title="Total do grupo",
            format="d"
        ),
        alt.Tooltip(
            "proporcao_graves:Q",
            title="Proporção de graves",
            format=".1%"
        )
    ]
)

textos = base_calor.mark_text(
    fontSize=18,
    fontWeight="bold",
    lineBreak="\n",
    lineHeight=26
).encode(
    text="rotulo:N",
    color=alt.condition(
        alt.datum.proporcao_graves >= 0.27,
        alt.value("white"),
        alt.value("#111111")
    )
)

mapa_calor = (
    celulas + textos
).properties(
    width=440,
    height=280,
    title=alt.TitleParams(
        text="Proporção de acidentes graves",
        subtitle=[
            "Por período e tipo de pista",
            "Dia: 06h até antes de 18h; Noite: demais horários.",
            "Rótulos: proporção e (graves / total do grupo).",
            "20.645 registros em pistas simples e duplas."
        ],
        anchor="start"
    )
).configure_view(
    stroke=None
)

with alt.renderers.enable("png", scale_factor=2):
    display(mapa_calor)

## Conclusão
Durante a noite os acidentes totais são menores do que durante o dia, entretanto ocorre proporcionalmente mais acidentes (registrados) no período da noite. Em ambos os momentos do dia, na pista simples temos uma maior ocorrência de acidentes graves, sendo que na pista simples a noite temos quase 1,7 vezes a quantidade de registros do que na pista dupla durante o dia.

# Como essa proporção varia entre condições meteorológicas, considerando cada estado?

In [ ]:
comparacao_pista_horario

In [ ]:
grupos=["uf","condicao_metereologica"]

acidentes_metereologicas_totais=sudeste_com_horarios.groupby(grupos)["id"].nunique().fillna(0).rename("total_acidentes").reset_index()
acidentes_metereologicas_graves=acidentes_graves.groupby(grupos)["id"].nunique().fillna(0).rename("acidentes_graves").reset_index()
comparacao_metereologica_estado=acidentes_metereologicas_totais.merge(acidentes_metereologicas_graves,on=grupos,how="left",validate="one_to_one").fillna(0)
comparacao_metereologica_estado["acidentes_graves"]=comparacao_metereologica_estado['acidentes_graves'].astype(int)
comparacao_metereologica_estado=comparacao_metereologica_estado[~comparacao_metereologica_estado["condicao_metereologica"].isin(["Ignorado"])].copy()
comparacao_metereologica_estado["proporcao_graves"]=comparacao_metereologica_estado["acidentes_graves"]/comparacao_metereologica_estado["total_acidentes"]
comparacao_metereologica_estado["rotulo"]=[(f"{linha.proporcao_graves:.1%}\n"f"({linha.acidentes_graves}/{linha.total_acidentes})").replace(".",",") for linha in comparacao_metereologica_estado.itertuples()]
comparacao_metereologica_estado.sort_values(by="proporcao_graves",ascending=False)

Na coluna "Condição meteorológica" havia algumas células classificadas como "ignorado", representando acidentes que não tiveram as condições devidamente registradas. Como isso não acrescenta nenhmuma informação na nossa análise, resolvemos retirar a fazer a observação sem esses dados.

In [ ]:
ordem_estados = ["ES", "MG", "RJ", "SP"]

ordem_condicoes = [
    "Céu Claro",
    "Sol",
    "Nublado",
    "Garoa/Chuvisco",
    "Chuva",
    "Nevoeiro/Neblina",
    "Vento",
    "Neve"
]

combinacoes_meteorologicas = pd.MultiIndex.from_product(
    [ordem_estados, ordem_condicoes],
    names=["uf", "condicao_metereologica"]
)

dados_mapa_meteorologico = (
    comparacao_metereologica_estado
    .set_index(["uf", "condicao_metereologica"])
    .reindex(combinacoes_meteorologicas)
    .reset_index()
)

dados_mapa_meteorologico["rotulo"] = (
    dados_mapa_meteorologico["rotulo"].fillna("Sem registros")
)

In [ ]:
base_meteorologica = alt.Chart(
    dados_mapa_meteorologico
).encode(
    x=alt.X(
        "uf:N",
        title="Estado",
        sort=ordem_estados,
        axis=alt.Axis(
            orient="top",
            labelAngle=0,
            labelPadding=8
        )
    ),
    y=alt.Y(
        "condicao_metereologica:N",
        title="Condição meteorológica",
        sort=ordem_condicoes,
        axis=alt.Axis(
            labelLimit=180,
            labelPadding=8
        )
    )
)

fundo_meteorologico = base_meteorologica.mark_rect(
    color="#e0e0e0",
    stroke="white",
    strokeWidth=2
)

celulas_meteorologicas = base_meteorologica.transform_filter(
    alt.datum.total_acidentes > 0
).mark_rect(
    stroke="white",
    strokeWidth=2
).encode(
    color=alt.Color(
        "proporcao_graves:Q",
        title="Proporção de graves",
        scale=alt.Scale(
            scheme="blues",
            domain=[0, 0.5]
        ),
        legend=alt.Legend(
            format=".0%",
            values=[0, 0.1, 0.2, 0.3, 0.4, 0.5]
        )
    ),
    tooltip=[
        alt.Tooltip("uf:N", title="Estado"),
        alt.Tooltip(
            "condicao_metereologica:N",
            title="Condição"
        ),
        alt.Tooltip(
            "acidentes_graves:Q",
            title="Acidentes graves",
            format="d"
        ),
        alt.Tooltip(
            "total_acidentes:Q",
            title="Total do grupo",
            format="d"
        ),
        alt.Tooltip(
            "proporcao_graves:Q",
            title="Proporção de graves",
            format=".1%"
        )
    ]
)

textos_meteorologicos = base_meteorologica.mark_text(
    fontSize=14,
    fontWeight="bold",
    lineBreak="\n",
    lineHeight=20,
    dy=-8
).encode(
    text="rotulo:N",
    color=alt.condition(
        alt.datum.proporcao_graves >= 0.35,
        alt.value("white"),
        alt.value("#111111")
    )
)

mapa_calor_meteorologico = (
    fundo_meteorologico
    + celulas_meteorologicas
    + textos_meteorologicos
).properties(
    width=600,
    height=alt.Step(65),
    title=alt.TitleParams(
        text="Gravidade dos acidentes por condição meteorológica e estado",
        subtitle=[
            "Sudeste — 2025 | Fonte: PRF",
            "Grave: pelo menos um morto ou um ferido grave.",
            "Rótulos: proporção e (graves / total do grupo).",
            "Cinza: sem registros na combinação. Categoria 'Ignorado' excluída."
        ],
        anchor="start"
    )
).configure_view(
    stroke=None
)

with alt.renderers.enable("png", scale_factor=2):
    display(mapa_calor_meteorologico)

## Conclusão
A proporção de acidentes graves variou entre as condições meteorológicas e os estados. Nas condições mais frequentes, Espírito Santo e Minas Gerais apresentaram proporções superiores às de Rio de Janeiro e São Paulo. O maior percentual foi observado sob nevoeiro/neblina no Espírito Santo, com 46,2%, mas corresponde a apenas 6 acidentes graves em 13 registros. Vale ressaltar que São Paulo se manteve abaixo dos 20% em todas as observações, enquanto a menor de Espírito Santo é 27,2%. 

# As diferenças de gravidade entre condições meteorológicas permanecem em comparações na mesma rodovia e em faixas horárias semelhantes?

In [ ]:
graves=(sudeste["mortos"]>0) | (sudeste["feridos_graves"]>0)
acidentes_graves_com_horario=sudeste_com_horarios[graves].copy()

grupos=["br","condicao_metereologica","faixa_horaria"]

clima_rodovia_horario_total = sudeste_com_horarios.groupby(grupos)["id"].nunique().fillna(0).rename("acidentes_total").reset_index()
clima_rodovia_horario_total["br"].unique()

rodovias_suspeitas=(clima_rodovia_horario_total["br"]==0) | (clima_rodovia_horario_total["br"]==40) | (clima_rodovia_horario_total["br"]==50)

In [ ]:
sudeste["br"].dtypes

## Tratamento de dados
Desde o momento em que importamos o dataset, a coluna "br" já é tida como um número inteiro (na primeira pergunta fizemos o tratamento dessa coluna, mas como estávamos analisando usando dados de outras fontes a verificação ocorria por pertencer à rodovia ou não dado a distância, portanto não fizemos essa alteração em nenhum dataset já que não nos seria necessário). Pesquisando os locais das rodovias no google maps usando a latitude e longitude, descobrimos que os acidentes registrados com "br" sendo 0 estão inconsistentes e não pertencem à uma rodovia em específico, portanto o código não permite identificar a rodovia para essa comparação, logo não iremos usar esses dados. Já o restante das células com 2 dígitos nas rodovias, foi percebido empiricamente (pesquisando algumas das localizações em aplicativos de gps) que as rodovias registradas são as mesmas, portanto era apenas um caso de padronização de escrita em números inteiros, o que excluia o dígito 0 inicial. Assim, iremos tratar esses dados no dataset final.

In [ ]:
clima_rodovia_horario_grave=acidentes_graves_com_horario.groupby(grupos)["id"].nunique().fillna(0).rename("acidentes_graves").reset_index()
comparacao_clima_rodovia_horario=clima_rodovia_horario_total.merge(clima_rodovia_horario_grave,on=grupos,how="left",validate="one_to_one").fillna(0)

In [ ]:
comparacao_clima_rodovia_horario=comparacao_clima_rodovia_horario[~comparacao_clima_rodovia_horario["br"].isin([0])]
comparacao_clima_rodovia_horario["br"] = (comparacao_clima_rodovia_horario["br"].astype(str).str.zfill(3))
comparacao_clima_rodovia_horario["proporcao"] = comparacao_clima_rodovia_horario["acidentes_graves"]/comparacao_clima_rodovia_horario["acidentes_total"]
comparacao_clima_rodovia_horario["acidentes_graves"]=comparacao_clima_rodovia_horario["acidentes_graves"].astype(int)
comparacao_clima_rodovia_horario["rotulo"]=([f"{linha.proporcao:.1%} ({linha.acidentes_graves}/{linha.acidentes_total})".replace(".",",") for linha in comparacao_clima_rodovia_horario.itertuples()])
comparacao_clima_rodovia_horario["condicao_metereologica"].unique()

comparacao_br_acidentes=comparacao_clima_rodovia_horario.copy()
#Essa cópia foi feita apenas para podermos usar esse dataset na resposta da próxima pergunta.

In [ ]:
ignorado=comparacao_clima_rodovia_horario["condicao_metereologica"]=="Ignorado"
print(f"Quantidade de registros classificados como ignorado: {comparacao_clima_rodovia_horario[ignorado]["acidentes_total"].sum()}")

Assim como foi feito antes, novamente iremos retirar da análise as linhas que na célula referente à coluna "condicao_metereologica" contém "Ignorado", pois essa categoria representa informação meteorológica desconhecida. Com isso, foram excluídos 271 registros dessa comparação.

In [ ]:
comparacao_clima_rodovia_horario=comparacao_clima_rodovia_horario[~comparacao_clima_rodovia_horario["condicao_metereologica"].isin(["Ignorado"])].copy()
comparacao_clima_rodovia_horario["condicao_metereologica"].unique()

In [ ]:
comparacao_clima_rodovia_horario

In [ ]:
comparacao_clima_rodovia_horario["acidentes_total"].describe()

A quantidade de acidentes varia bastante entre as 236 combinações de rodovia, condição meteorológica e período. Pelo menos metade desses grupos possui até 11 acidentes, enquanto a média é de aproximadamente 97,48. O maior grupo reúne 2.262 registros na BR-116, sob Céu Claro, durante o Dia. Essa diferença mostra que algumas proporções são calculadas com centenas de ocorrências, enquanto outras dependem de poucos registros e precisam ser interpretadas com maior cuidado.

Para facilitar a leitura dos gráficos e concentrar a comparação nas rodovias com maior disponibilidade de registros, selecionamos as cinco BRs com mais acidentes nesse recorte: BR-116, BR-101, BR-381, BR-040 e BR-262. Juntas, elas reúnem 19.016 dos 23.005 registros com rodovia identificada e condição meteorológica conhecida, aproximadamente 82,66%. Mesmo nessas rodovias, alguns grupos permanecem pequenos. Por isso, apresentamos a quantidade de acidentes graves e o total de cada grupo nos rótulos, e restringimos as conclusões às rodovias selecionadas.

In [ ]:
acidentes_por_rodovia=comparacao_clima_rodovia_horario.groupby("br")["acidentes_total"].sum().rename("acidentes totais").fillna(0).reset_index()
acidentes_por_rodovia.sort_values(by="acidentes totais",ascending=False).head(5)

In [ ]:
comparacao_clima_rodovia_horario_top5=comparacao_clima_rodovia_horario[comparacao_clima_rodovia_horario["br"].isin(["116","101","381","040","262"])].copy()
comparacao_clima_rodovia_horario_top5

In [ ]:
dados_pontos = comparacao_clima_rodovia_horario_top5.copy()

ordem_brs = ["116", "101", "381", "040", "262"]
ordem_periodos = ["Dia", "Noite"]

ordem_clima = [
    "Céu Claro",
    "Sol",
    "Nublado",
    "Garoa/Chuvisco",
    "Chuva",
    "Nevoeiro/Neblina",
    "Vento"
]

In [ ]:
base_paineis = alt.Chart(dados_pontos).encode(
    x=alt.X(
        "proporcao:Q",
        title="Proporção de acidentes graves",
        scale=alt.Scale(domain=[0, 1]),
        axis=alt.Axis(format=".0%", tickCount=6)
    ),
    y=alt.Y(
        "condicao_metereologica:N",
        title=None,
        scale=alt.Scale(domain=ordem_clima),
        axis=alt.Axis(labelLimit=170)
    ),
    yOffset=alt.YOffset(
        "faixa_horaria:N",
        sort=ordem_periodos,
        scale=alt.Scale(domain=ordem_periodos)
    )
)

pontos_paineis = base_paineis.mark_point(
    filled=True,
    opacity=1,
    size=70
).encode(
    color=alt.Color(
        "faixa_horaria:N",
        title="Período",
        scale=alt.Scale(
            domain=ordem_periodos,
            range=["#0072B2", "#D55E00"]
        ),
        legend=alt.Legend(orient="top")
    ),
    shape=alt.Shape(
        "faixa_horaria:N",
        title="Período",
        scale=alt.Scale(
            domain=ordem_periodos,
            range=["circle", "square"]
        ),
        legend=alt.Legend(orient="top")
    ),
    tooltip=[
        alt.Tooltip("br:N", title="BR"),
        alt.Tooltip("condicao_metereologica:N", title="Condição"),
        alt.Tooltip("faixa_horaria:N", title="Período"),
        alt.Tooltip("acidentes_graves:Q", title="Graves", format="d"),
        alt.Tooltip("acidentes_total:Q", title="Total", format="d"),
        alt.Tooltip("proporcao:Q", title="Proporção", format=".1%")
    ]
)

rotulos_direita = base_paineis.transform_filter(
    alt.datum.proporcao <= 0.5
).mark_text(
    align="left",
    dx=6,
    fontSize=10,
    color="#333333"
).encode(
    text="rotulo:N"
)

rotulos_esquerda = base_paineis.transform_filter(
    alt.datum.proporcao > 0.5
).mark_text(
    align="right",
    dx=-6,
    fontSize=10,
    color="#333333"
).encode(
    text="rotulo:N"
)

grafico_paineis = (
    pontos_paineis + rotulos_direita + rotulos_esquerda
).properties(
    width=330,
    height=290
).facet(
    facet=alt.Facet(
        "br:N",
        title="Rodovia federal",
        sort=ordem_brs,
        header=alt.Header(
            labelFontSize=14,
            labelFontWeight="bold"
        )
    ),
    columns=2
).properties(
    spacing=25,
    title=alt.TitleParams(
        text="Gravidade por condição meteorológica e período",
        subtitle=[
            "Sudeste — 2025 | Fonte: PRF | Cinco BRs com mais registros no recorte",
            "Rótulos: proporção (graves / total do grupo).",
            "Ausência de ponto: sem registros na combinação.",
        ],
        anchor="start"
    )
).resolve_scale(
    x="shared",
    y="shared"
).resolve_axis(
    x="independent"
).configure_view(
    stroke=None
)

with alt.renderers.enable("png", scale_factor=2):
    display(grafico_paineis)

Em todos os registros que contêm uma quantidade razoável de acidentes, é observável que a proporção fica entre 12% a 30% em 4 das 5 rodovias. Não parece haver uma distinção muito grande na proporção entre dia e noite, e nem mesmo entre as condições meteorológicas. Entretanto, na br 262 a proporção de acidentes graves no geral é maior que nas outras rodovias, sendo 24,4% de acidentes graves durante a chuva a menor porcentagem, e 40,2% durante clima nublado a maior porcentagem (isto é, com uma quantidade razoável de acidentes registrados).

Para aprofundar a comparação, selecionamos Céu Claro e Chuva, duas condições frequentes e presentes nas cinco rodovias. Nesse recorte, todos os grupos possuem pelo menos 45 acidentes registrados. O gráfico de pontos conectados destaca a direção e o tamanho das diferenças entre essas duas condições, complementando a visualização mais ampla das demais categorias.

In [ ]:
dados_conectados = dados_pontos[
    dados_pontos["condicao_metereologica"].isin(
        ["Céu Claro", "Chuva"]
    )
].copy()

base_conectados = alt.Chart(dados_conectados).encode(
    x=alt.X(
        "proporcao:Q",
        title="Proporção de acidentes graves",
        scale=alt.Scale(domain=[0, 0.5]),
        axis=alt.Axis(format=".0%", tickCount=6)
    ),
    y=alt.Y(
        "br:N",
        title="BR",
        sort=ordem_brs
    )
)

ligacoes_clima = base_conectados.mark_line(
    color="#aaaaaa",
    strokeWidth=2
).encode(
    detail="br:N",
    order=alt.Order(
        "condicao_metereologica:N",
        sort="ascending"
    )
)

pontos_clima = base_conectados.mark_point(
    filled=True,
    opacity=1,
    size=110
).encode(
    color=alt.Color(
        "condicao_metereologica:N",
        title="Condição",
        scale=alt.Scale(
            domain=["Céu Claro", "Chuva"],
            range=["#009E73", "#AA3377"]
        ),
        legend=alt.Legend(orient="top")
    ),
    shape=alt.Shape(
        "condicao_metereologica:N",
        title="Condição",
        scale=alt.Scale(
            domain=["Céu Claro", "Chuva"],
            range=["circle", "diamond"]
        ),
        legend=alt.Legend(orient="top")
    ),
    tooltip=[
        alt.Tooltip("br:N", title="BR"),
        alt.Tooltip("faixa_horaria:N", title="Período"),
        alt.Tooltip("condicao_metereologica:N", title="Condição"),
        alt.Tooltip("acidentes_graves:Q", title="Graves", format="d"),
        alt.Tooltip("acidentes_total:Q", title="Total", format="d"),
        alt.Tooltip("proporcao:Q", title="Proporção", format=".1%")
    ]
)

rotulos_ceu = base_conectados.transform_filter(
    alt.datum.condicao_metereologica == "Céu Claro"
).mark_text(
    dy=-18,
    fontSize=11,
    color="#007B59"
).encode(
    text="rotulo:N"
)

rotulos_chuva = base_conectados.transform_filter(
    alt.datum.condicao_metereologica == "Chuva"
).mark_text(
    dy=18,
    fontSize=11,
    color="#AA3377"
).encode(
    text="rotulo:N"
)

grafico_conectados = (
    ligacoes_clima + pontos_clima + rotulos_ceu + rotulos_chuva
).properties(
    width=330,
    height=alt.Step(75)
).facet(
    column=alt.Column(
        "faixa_horaria:N",
        title=None,
        sort=ordem_periodos,
        header=alt.Header(
            labelFontSize=14,
            labelFontWeight="bold"
        )
    )
).properties(
    title=alt.TitleParams(
        text="Gravidade: Chuva e Céu Claro na mesma BR e período",
        subtitle=[
            "Sudeste — 2025 | Fonte: PRF | Cinco BRs com mais registros no recorte",
            "Rótulos: proporção (graves / total do grupo).",
            "O segmento liga as duas condições na mesma rodovia e no mesmo período."
        ],
        anchor="start"
    )
).resolve_scale(
    x="shared",
    y="shared"
).configure_view(
    stroke=None
)

with alt.renderers.enable("png", scale_factor=2):
    display(grafico_conectados)

No geral, entre acidentes que ocorrem em céu claro e em chuva, a maior porcentagem dos acidentes graves registrados acontecem em céu claro. No geral, a proporção desses acidentes aumentam durante a noite, sendo que a maior discrepância de porcentagem entre céu claro e chuva ocorreu na br 262 no período da noite.

## Conclusão
Nas cinco rodovias analisadas, foram observadas diferenças na proporção de acidentes graves entre condições meteorológicas, mesmo ao separar BR e período. Na comparação entre Chuva e Céu Claro, a proporção foi maior sob céu claro em nove das dez combinações de rodovia e período; a exceção foi a BR-101 durante o dia. Outras condições apresentaram percentuais elevados, mas frequentemente baseados em poucos registros, limitando sua generalização.

# As rodovias com mais acidentes registrados também apresentam maior proporção de ocorrências graves?

## Tratamento de dados
Foi usado um dataset intermediário na pergunta anterior para responder a atual. Como ainda estamos analisando as rodovias iremos continuar sem incluir a br 0, pelos mesmos motivos anteriores. Entretanto, iremos manter os acidentes com condição meteorológica classificada como "Ignorado", pois isso não afeta na análise que iremos trabalhar.

In [ ]:
br_acidentes_total=comparacao_br_acidentes.groupby("br")["acidentes_total"].sum().fillna(0).rename("Acidentes_totais").reset_index()
br_acidentes_graves=comparacao_br_acidentes.groupby("br")["acidentes_graves"].sum().fillna(0).rename("Acidentes_graves").reset_index()
comparacao_br_acidentes_graves=br_acidentes_total.merge(br_acidentes_graves,on="br",how="left",validate="one_to_one").fillna(0)
comparacao_br_acidentes_graves["proporcao"]=comparacao_br_acidentes_graves["Acidentes_graves"]/comparacao_br_acidentes_graves["Acidentes_totais"]
comparacao_br_acidentes_graves["rotulo"]=[(f"{linha.proporcao:.1%}"f"({linha.Acidentes_graves}/{linha.Acidentes_totais})")for linha in comparacao_br_acidentes_graves.itertuples()]
comparacao_br_acidentes_graves=comparacao_br_acidentes_graves.sort_values(by="proporcao",ascending=False)
comparacao_br_acidentes_graves["ranking_proporção"]=[(f"{br}")for br in range(1,comparacao_br_acidentes_graves["br"].nunique() + 1)]
comparacao_br_acidentes_graves["ranking_proporção"]=comparacao_br_acidentes_graves["ranking_proporção"].astype(int)

In [ ]:
comparacao_br_acidentes_graves.sort_values(by="Acidentes_totais",ascending=False)

In [ ]:
dados_dispersao = comparacao_br_acidentes_graves.copy()

limite_destaque = 10

dados_dispersao["grupo_registros"] = [
    "Menos de 10 registros"
    if total < limite_destaque
    else "10 ou mais registros"
    for total in dados_dispersao["Acidentes_totais"]
]

dados_dispersao["identificacao"] = [
    f"BR-{linha.br}: {linha.rotulo}".replace(".", ",")
    for linha in dados_dispersao.itertuples()
]

deslocamentos = {
    "040": 16,
    "116": 16,
    "050": 16,
    "267": 16,
    "259": -28
}

dados_dispersao["dy_rotulo"] = (
    dados_dispersao["br"].map(deslocamentos).fillna(-12)
)

In [ ]:
base_dispersao = alt.Chart(dados_dispersao).encode(
    x=alt.X(
        "Acidentes_totais:Q",
        title="Total de acidentes registrados (escala logarítmica)",
        scale=alt.Scale(
            type="log",
            base=10,
            domain=[0.8, 10000]
        ),
        axis=alt.Axis(
            values=[1, 10, 100, 1000, 10000],
            format=".0f"
        )
    ),
    y=alt.Y(
        "proporcao:Q",
        title="Proporção de acidentes graves",
        scale=alt.Scale(domain=[0, 1.08]),
        axis=alt.Axis(
            values=[0, 0.2, 0.4, 0.6, 0.8, 1],
            format=".0%"
        )
    )
)

pontos_dispersao = base_dispersao.mark_point(
    filled=True,
    opacity=1,
    size=110,
    stroke="white",
    strokeWidth=1
).encode(
    color=alt.Color(
        "grupo_registros:N",
        title="Quantidade de registros",
        scale=alt.Scale(
            domain=[
                "10 ou mais registros",
                "Menos de 10 registros"
            ],
            range=["#0072B2", "#D55E00"]
        ),
        legend=alt.Legend(orient="top")
    ),
    shape=alt.Shape(
        "grupo_registros:N",
        title="Quantidade de registros",
        scale=alt.Scale(
            domain=[
                "10 ou mais registros",
                "Menos de 10 registros"
            ],
            range=["circle", "diamond"]
        ),
        legend=alt.Legend(orient="top")
    ),
    tooltip=[
        alt.Tooltip("br:N", title="BR"),
        alt.Tooltip(
            "Acidentes_totais:Q",
            title="Total de acidentes",
            format="d"
        ),
        alt.Tooltip(
            "Acidentes_graves:Q",
            title="Acidentes graves",
            format="d"
        ),
        alt.Tooltip(
            "proporcao:Q",
            title="Proporção de graves",
            format=".1%"
        ),
        alt.Tooltip("grupo_registros:N", title="Grupo")
    ]
)

rotulos_brs = base_dispersao.transform_filter(
    alt.datum.Acidentes_totais >= limite_destaque
).mark_text(
    fontSize=11,
    color="#333333",
    dy=alt.ExprRef(expr="datum.dy_rotulo")
).encode(
    text="br:N"
)

rotulos_poucos = base_dispersao.transform_filter(
    (alt.datum.Acidentes_totais < limite_destaque)
    & (alt.datum.br != "265")
    & (alt.datum.br != "485")
).mark_text(
    align="left",
    dx=9,
    dy=-12,
    fontSize=11,
    fontWeight="bold",
    color="#A34500"
).encode(
    text="identificacao:N"
)

rotulo_coincidente = base_dispersao.transform_filter(
    alt.datum.br == "265"
).mark_text(
    align="left",
    dx=9,
    dy=-12,
    fontSize=11,
    fontWeight="bold",
    color="#A34500"
).encode(
    text=alt.value("BR-265 e BR-485: 0% (0/2 cada)")
)

grafico_dispersao = (
    pontos_dispersao
    + rotulos_brs
    + rotulos_poucos
    + rotulo_coincidente
).properties(
    width=780,
    height=470,
    title=alt.TitleParams(
        text="Mais acidentes registrados significam maior proporção de graves?",
        subtitle=[
            "Sudeste — 2025 | Fonte: PRF | 25 rodovias identificadas",
            "Cada ponto representa uma BR. Os números junto aos pontos identificam as rodovias.",
            "Laranja: menos de 10 registros; corte apenas para destaque visual.",
            "As BR-265 e BR-485 possuem a mesma posição: 0 graves em 2 registros cada."
        ],
        anchor="start"
    )
).configure_view(
    stroke=None
)

with alt.renderers.enable("png", scale_factor=2):
    display(grafico_dispersao)

#### Descrição da visualização:
O eixo horizontal utiliza uma escala logarítmica de base 10 para representar o total de acidentes por rodovia. Nessa escala, distâncias iguais correspondem a multiplicações iguais: de 10 para 100 há a mesma distância visual que de 100 para 1.000. Essa escolha facilita a visualização de rodovias com quantidades muito diferentes de registros, sem alterar os valores dos dados. Portanto, não tente estimar a quantidade de acidentes registrados da cada rodovia visualmente, nem mesmo comparar a proporção entre as rodovias, esse gráfico apenas passa a informação de ranking das rodovias em relação à proporção indivual de acidentes graves. O eixo vertical mantém uma escala linear para a proporção de acidentes graves.

Se achar pertinente ter ciência da quantidade total de acidentes por rodovia, logo acima existe um dataset com as devidas informações.

## Conclusão
As rodovias com mais acidentes registrados não apresentaram necessariamente as maiores proporções de ocorrências graves. A BR-116 reuniu o maior volume, com 6.459 acidentes e 22,4% de graves, enquanto a BR-262 apresentou 34,2% entre seus 1.484 registros. Portanto, volume de ocorrências e proporção de gravidade representam aspectos diferentes da distribuição dos acidentes. Percentuais extremos exigem cuidado: os 100% da BR-484 correspondem a apenas um registro, o que limita sua generalização e não permite classificá-la como a rodovia mais perigosa.